# 01: Data Ingestion

---
## 0. Setup — paths & imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import time
from pathlib import Path

BASE_DIR    = Path().resolve().parent
RAW_DIR     = BASE_DIR / 'data' / 'raw'
PROC_DIR    = BASE_DIR / 'data' / 'processed'
REPORTS_DIR = BASE_DIR / 'reports'

print('Base directory :', BASE_DIR)
print('Raw data dir   :', RAW_DIR)
print('Pandas version :', pd.__version__)
print('NumPy version  :', np.__version__)

Base directory : /home/l011yp0p/data/intership/bluestock_mf_capstone
Raw data dir   : /home/l011yp0p/data/intership/bluestock_mf_capstone/data/raw
Pandas version : 2.2.2
NumPy version  : 1.26.4


---
## 1. Load all 10 CSV datasets

In [2]:
datasets = {}

csv_files = sorted(RAW_DIR.glob("*.csv"))

if not csv_files:
    print("No CSV files found in data/raw/")
else:
    print(f"Found {len(csv_files)} CSV files\n")

for fpath in csv_files:
    try:
        df = pd.read_csv(fpath, low_memory=False)
        datasets[fpath.name] = df
        print(f"{fpath.name:<35}")

    except Exception as e:
        print(f"{fpath.name:<35} ERROR: {e}")

print(f"\nLoaded {len(datasets)} dataset(s).")

Found 16 CSV files

01_fund_master.csv                 
02_nav_history.csv                 
03_aum_by_fund_house.csv           
04_monthly_sip_inflows.csv         
05_category_inflows.csv            
06_industry_folio_count.csv        
07_scheme_performance.csv          
08_investor_transactions.csv       
09_portfolio_holdings.csv          
10_benchmark_indices.csv           
nav_118632_Nippon_LargeCap_Direct.csv
nav_119092_Axis_Bluechip_Direct.csv
nav_119551_SBI_Bluechip_Direct.csv 
nav_120503_ICICI_Bluechip_Direct.csv
nav_120841_Kotak_Bluechip_Direct.csv
nav_125497_HDFC_Top100_Direct.csv  

Loaded 16 dataset(s).


### 1.1 Inspect each dataset — dtypes & head

In [3]:
def inspect_all_datasets(datasets):

    for filename, df in datasets.items():

        print("\n" + "=" * 50)
        print(f"DATASET: {filename}")
        print("=" * 50)

        print(f"Shape: {df.shape}")

        print("\nDtypes:")
        print(df.dtypes)

        print("\nNull Counts:")
        print(df.isnull().sum())

        print("\nHead:")
        display(df.head())

inspect_all_datasets(datasets)


DATASET: 01_fund_master.csv
Shape: (40, 15)

Dtypes:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

Null Counts:
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

Head:


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02



DATASET: 02_nav_history.csv
Shape: (46000, 3)

Dtypes:
amfi_code      int64
date          object
nav          float64
dtype: object

Null Counts:
amfi_code    0
date         0
nav          0
dtype: int64

Head:


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692



DATASET: 03_aum_by_fund_house.csv
Shape: (90, 5)

Dtypes:
date               object
fund_house         object
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object

Null Counts:
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64

Head:


...

,date,nav
0,01-06-2026,103.0948
1,31-05-2026,104.3083
2,29-05-2026,104.3129
3,27-05-2026,105.9296
4,26-05-2026,105.5115



DATASET: nav_120841_Kotak_Bluechip_Direct.csv
Shape: (3301, 2)

Dtypes:
date     object
nav     float64
dtype: object

Null Counts:
date    0
nav     0
dtype: int64

Head:


,date,nav
0,01-06-2026,246.9814
1,29-05-2026,248.6673
2,27-05-2026,251.2273
3,26-05-2026,251.2702
4,25-05-2026,249.2266



DATASET: nav_125497_HDFC_Top100_Direct.csv
Shape: (3091, 2)

Dtypes:
date     object
nav     float64
dtype: object

Null Counts:
date    0
nav     0
dtype: int64

Head:


,date,nav
0,01-06-2026,192.3195
1,31-05-2026,193.6836
2,29-05-2026,193.6848
3,27-05-2026,195.0501
4,26-05-2026,194.2258


In [89]:
# Summary table for all datasets 
rows = []
for name, df in datasets.items():
    rows.append({
        'File'         : name,
        'Rows'         : df.shape[0],
        'Cols'         : df.shape[1],
        'Nulls %'      : round(df.isnull().mean().mean() * 100, 2),
        'Duplicates'   : int(df.duplicated().sum()),
    })
pd.DataFrame(rows)

,File,Rows,Cols,Nulls %,Duplicates
0,01_fund_master.csv,40,15,0.00,0
1,02_nav_history.csv,46000,3,0.00,0
2,03_aum_by_fund_house.csv,90,5,0.00,0
3,04_monthly_sip_inflows.csv,48,6,4.17,0
4,05_category_inflows.csv,144,3,0.00,0
5,06_industry_folio_count.csv,21,6,0.00,0
6,07_scheme_performance.csv,40,19,0.00,0
7,08_investor_transactions.csv,32778,13,0.00,0
8,09_portfolio_holdings.csv,322,8,0.00,0
9,10_benchmark_indices.csv,8050,3,0.00,0


---
## 2. Fetch live NAV from mfapi.in

In [ ]:
from pathlib import Path
import pandas as pd
import requests

SCHEMES = {
    125497: "HDFC_Top100_Direct",
    119551: "SBI_Bluechip_Direct",
    120503: "ICICI_Bluechip_Direct",
    118632: "Nippon_LargeCap_Direct",
    119092: "Axis_Bluechip_Direct",
    120841: "Kotak_Bluechip_Direct",
}

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

for code, name in SCHEMES.items():

    try:
        url = f"https://api.mfapi.in/mf/{code}"

        response = requests.get(url, timeout=15)
        response.raise_for_status()

        payload = response.json()

        if "data" not in payload:
            print(f"No NAV data found for {name}")
            continue

        df = pd.DataFrame(payload["data"])
        output_file = RAW_DIR / f"nav_{code}_{name}.csv"
        df.to_csv(output_file, index=False)
        print(f"Saved {output_file.name} ({len(df):,} rows)")

    except Exception as e:
        print(f"Failed {name}: {e}")

print("\nNAV fetch completed.")

Saved nav_125497_HDFC_Top100_Direct.csv (3,091 rows)
Saved nav_119551_SBI_Bluechip_Direct.csv (3,236 rows)
Saved nav_120503_ICICI_Bluechip_Direct.csv (3,307 rows)
Saved nav_118632_Nippon_LargeCap_Direct.csv (3,298 rows)
Saved nav_119092_Axis_Bluechip_Direct.csv (3,565 rows)
Saved nav_120841_Kotak_Bluechip_Direct.csv (3,301 rows)

NAV fetch completed.


---
## 3. Explore fund_master

In [84]:
if '01_fund_master.csv' in datasets:

    fm = datasets['01_fund_master.csv']

    print("FUND MASTER OVERVIEW")
    print("=" * 80)

    display(fm.head(3))

    columns_to_explore = {
        "Fund Houses": "fund_house",
        "Categories": "category",
        "Sub-Categories": "sub_category",
        "Risk Categories": "risk_category"
    }

    for label, col in columns_to_explore.items():

        unique_values = sorted(fm[col].dropna().unique())

        result_df = pd.DataFrame({
            label: unique_values
        })

        print(f"\n{label}")
        print(f"Total Unique Values: {len(unique_values)}")

        display(result_df)

    print("\nAMFI SCHEME CODE STRUCTURE")
    print("=" * 80)

    print("Sample AMFI Codes:")
    display(fm[['amfi_code', 'scheme_name']].head(10))

    print(
        "\nAMFI codes are unique numeric identifiers assigned "
        "to each mutual fund scheme by AMFI."
    )

    print(f"\nTotal Unique AMFI Codes: {fm['amfi_code'].nunique():,}")

FUND MASTER OVERVIEW


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03



Fund Houses
Total Unique Values: 10


,Fund Houses
0,Aditya Birla Sun Life MF
1,Axis Mutual Fund
2,DSP Mutual Fund
3,HDFC Mutual Fund
4,ICICI Prudential MF
5,Kotak Mahindra MF
6,Mirae Asset MF
7,Nippon India MF
8,SBI Mutual Fund
9,UTI Mutual Fund



Categories
Total Unique Values: 2


...


Risk Categories
Total Unique Values: 5


,Risk Categories
0,High
1,Low
2,Moderate
3,Moderately High
4,Very High



AMFI SCHEME CODE STRUCTURE
Sample AMFI Codes:


,amfi_code,scheme_name
0,119551,SBI Bluechip Fund - Regular Plan - Growth
1,119552,SBI Bluechip Fund - Direct Plan - Growth
2,119598,SBI Small Cap Fund - Regular Plan - Growth
3,119599,SBI Small Cap Fund - Direct Plan - Growth
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth
5,100016,HDFC Top 100 Fund - Regular Plan - Growth
6,125497,HDFC Top 100 Fund - Direct Plan - Growth
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
8,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth
9,100025,HDFC Short Term Debt Fund - Regular - Growth



AMFI codes are unique numeric identifiers assigned to each mutual fund scheme by AMFI.

Total Unique AMFI Codes: 40


---
## 4. Validating AMFI codes

In [86]:
if '01_fund_master.csv' in datasets and '02_nav_history.csv' in datasets:
    fm  = datasets['01_fund_master.csv']
    nav = datasets['02_nav_history.csv']

    # Adjust these column names if yours differ
    FM_CODE_COL  = 'amfi_code'   # scheme_code column in 01_fund_master
    NAV_CODE_COL = 'amfi_code'   # scheme_code column in 02_nav_history

    fm_codes  = set(fm[FM_CODE_COL].dropna().astype(int).unique())
    nav_codes = set(nav[NAV_CODE_COL].dropna().astype(int).unique())

    missing_in_nav = fm_codes - nav_codes
    extra_in_nav   = nav_codes - fm_codes
    matched        = fm_codes & nav_codes

    print('AMFI CODE VALIDATION')
    print('-' * 40)
    print(f'  01_fund_master  total codes       : {len(fm_codes):,}')
    print(f'  02_nav_history  total codes       : {len(nav_codes):,}')
    print(f'  Matched (present in both)      : {len(matched):,}')
    print(f'  In 01_fund_master, missing in NAV : {len(missing_in_nav):,}')
    print(f'  In NAV, missing in 01_fund_master : {len(extra_in_nav):,}')

    if missing_in_nav:
        print(f'\n  Missing codes (first 20): {sorted(missing_in_nav)[:20]}')
    else:
        print('\n All 01_fund_master codes are present in 02_nav_history.')
else:
    print('[INFO] 01_fund_master.csv or 02_nav_history.csv not loaded — skipping validation.')

AMFI CODE VALIDATION
----------------------------------------
  01_fund_master  total codes       : 40
  02_nav_history  total codes       : 40
  Matched (present in both)      : 40
  In 01_fund_master, missing in NAV : 0
  In NAV, missing in 01_fund_master : 0

 All 01_fund_master codes are present in 02_nav_history.


---
## 5. Data Quality Summary

In [87]:
rows = []
for name, df in datasets.items():
    high_null_cols = df.columns[df.isnull().mean() > 0.2].tolist()
    rows.append({
        'File'            : name,
        'Rows'            : f'{df.shape[0]:,}',
        'Cols'            : df.shape[1],
        'Avg null %'      : f'{df.isnull().mean().mean()*100:.1f}%',
        'Duplicate rows'  : df.duplicated().sum(),
        'High-null cols'  : ', '.join(high_null_cols) if high_null_cols else '—',
    })

quality_df = pd.DataFrame(rows)
quality_df

,File,Rows,Cols,Avg null %,Duplicate rows,High-null cols
0,01_fund_master.csv,40,15,0.0%,0,—
1,02_nav_history.csv,"46,000",3,0.0%,0,—
2,03_aum_by_fund_house.csv,90,5,0.0%,0,—
3,04_monthly_sip_inflows.csv,48,6,4.2%,0,yoy_growth_pct
4,05_category_inflows.csv,144,3,0.0%,0,—
5,06_industry_folio_count.csv,21,6,0.0%,0,—
6,07_scheme_performance.csv,40,19,0.0%,0,—
7,08_investor_transactions.csv,"32,778",13,0.0%,0,—
8,09_portfolio_holdings.csv,322,8,0.0%,0,—
9,10_benchmark_indices.csv,"8,050",3,0.0%,0,—


In [88]:
# Exporting quality summary to reports/
quality_df.to_csv(REPORTS_DIR / 'data_quality_summary.csv', index=False)
print('Data quality summary saved to reports/data_quality_summary.csv')

Data quality summary saved to reports/data_quality_summary.csv
